# Day 064 — Solution: Capstone Build I

In [ ]:
_APP_SRC = '"""writing_assistant.py — Day 064: AI Writing Assistant MVP.\n\nSetup:\n  pip install fastapi "uvicorn[standard]" ollama\n  ollama pull llama3.2\n\nRun:\n  uvicorn writing_assistant:app --reload\nDocs:\n  http://localhost:8000/docs\n"""\nimport os\nimport re\nimport secrets\nimport time\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nAPP_VER = "1.0.0"\nMODEL   = os.environ.get("MODEL", "llama3.2")\n\n# ── plan configuration ────────────────────────────────────────────────────────\n\nDAILY_LIMITS = {"free": 5, "pro": 500, "enterprise": float("inf")}\n\nFEATURE_MATRIX = {\n    "free": {"basic_generate", "view_history"},\n    "pro":  {"basic_generate", "view_history", "improve_text", "export"},\n}\n\nTEMPLATES = {\n    "email":     "Write a {tone} email to {recipient} about {topic}.",\n    "tweet":     "Write a {tone} tweet about {topic} in under 280 characters.",\n    "summary":   "Write a concise {length}-sentence summary of: {content}",\n    "blog_intro": "Write an engaging blog introduction about {topic} for a {audience} audience.",\n}\n\n\ndef check_feature_access(plan: str, feature: str) -> bool:\n    return feature in FEATURE_MATRIX.get(plan, set())\n\n\ndef check_rate_limit(usage_count: int, plan: str) -> tuple[bool, str]:\n    limit = DAILY_LIMITS.get(plan, 0)\n    if usage_count >= limit:\n        return False, f"Daily limit reached for {plan!r} plan ({usage_count}/{int(limit)})"\n    return True, ""\n\n\n# ── content store ─────────────────────────────────────────────────────────────\n\nclass ContentStore:\n    def __init__(self):\n        self._store: dict = {}\n\n    def add(self, user_id: str, prompt: str, content: str) -> str:\n        cid = secrets.token_urlsafe(8)\n        self._store[cid] = {\n            "content_id": cid,\n            "user_id":    user_id,\n            "prompt":     prompt,\n            "content":    content,\n            "created_at": datetime.utcnow().isoformat() + "Z",\n        }\n        return cid\n\n    def get(self, content_id: str) -> dict | None:\n        return self._store.get(content_id)\n\n    def list_user(self, user_id: str) -> list[dict]:\n        return [v for v in self._store.values() if v["user_id"] == user_id]\n\n    def count(self, user_id: str) -> int:\n        return sum(1 for v in self._store.values() if v["user_id"] == user_id)\n\n\n# ── template engine ───────────────────────────────────────────────────────────\n\ndef render_template(template_str: str, **vars) -> str:\n    required = set(re.findall(r\'\\{(\\w+)\\}\', template_str))\n    missing  = required - set(vars.keys())\n    if missing:\n        raise ValueError(f"Missing template variables: {missing}")\n    result = template_str\n    for key, value in vars.items():\n        result = result.replace(f"{{{key}}}", str(value))\n    return result\n\n\n# ── FastAPI app ───────────────────────────────────────────────────────────────\n\ndef build_api(process_fn=None, initial_plan: str = "free",\n              initial_usage: int = 0) -> FastAPI:\n    app   = FastAPI(title="AI Writing Assistant", version=APP_VER)\n    store = ContentStore()\n    state = {"plan": initial_plan, "usage": initial_usage}\n\n    class GenerateRequest(BaseModel):\n        prompt:  str = Field(min_length=1)\n        user_id: str = Field(min_length=1)\n\n    class TemplateRequest(BaseModel):\n        template: str = Field(min_length=1)\n        vars:     dict = {}\n        user_id:  str = Field(min_length=1)\n\n    class ImproveRequest(BaseModel):\n        text:    str = Field(min_length=1)\n        user_id: str = Field(min_length=1)\n\n    @app.get("/health")\n    def health():\n        return {"status": "ok", "timestamp": datetime.utcnow().isoformat() + "Z",\n                "version": APP_VER}\n\n    @app.get("/plan")\n    def get_plan():\n        lim = DAILY_LIMITS.get(state["plan"], 0)\n        return {"plan": state["plan"], "usage_today": state["usage"],\n                "limit": lim if lim != float("inf") else -1}\n\n    @app.get("/templates")\n    def list_templates():\n        return {"templates": list(TEMPLATES.keys())}\n\n    @app.post("/generate")\n    def generate(req: GenerateRequest):\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        answer = (process_fn(req.prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": req.prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, req.prompt, answer)\n        return {"content_id": cid, "content": answer, "user_id": req.user_id}\n\n    @app.post("/generate/template")\n    def generate_from_template(req: TemplateRequest):\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        tmpl = TEMPLATES.get(req.template)\n        if tmpl is None:\n            raise HTTPException(400, f"Unknown template: {req.template!r}")\n        try:\n            prompt = render_template(tmpl, **req.vars)\n        except ValueError as e:\n            raise HTTPException(400, str(e))\n        answer = (process_fn(prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, prompt, answer)\n        return {"content_id": cid, "content": answer,\n                "template": req.template, "user_id": req.user_id}\n\n    @app.post("/improve")\n    def improve(req: ImproveRequest):\n        if not check_feature_access(state["plan"], "improve_text"):\n            raise HTTPException(403, "improve_text requires a pro plan")\n        ok, reason = check_rate_limit(state["usage"], state["plan"])\n        if not ok:\n            raise HTTPException(429, reason)\n        prompt = f"Improve this text for clarity and style:\\n\\n{req.text}"\n        answer = (process_fn(prompt) if process_fn\n                  else ollama.chat(model=MODEL,\n                                   messages=[{"role": "user",\n                                              "content": prompt}])\n                  ["message"]["content"])\n        state["usage"] += 1\n        cid = store.add(req.user_id, prompt, answer)\n        return {"content_id": cid, "original": req.text,\n                "improved": answer, "user_id": req.user_id}\n\n    @app.get("/history/{user_id}")\n    def get_history(user_id: str):\n        items = store.list_user(user_id)\n        return {"user_id": user_id, "count": len(items), "items": items}\n\n    @app.get("/content/{content_id}")\n    def get_content(content_id: str):\n        item = store.get(content_id)\n        if item is None:\n            raise HTTPException(404, f"Content {content_id!r} not found")\n        return item\n\n    return app\n\n\napp = build_api()\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('writing_assistant.py').write_text(_APP_SRC)
print('writing_assistant.py written.')

In [ ]:
# inline tests — no Ollama needed
import re, secrets
from datetime import datetime
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# ─── plan_mvp ────────────────────────────────────────────────────────────────
def plan_mvp(features, free_count, pro_count):
    free    = features[:free_count]
    pro     = features[free_count: free_count + pro_count]
    backlog = features[free_count + pro_count:]
    return {"free_tier": free, "pro_tier": pro, "backlog": backlog,
            "summary": f"{len(free)} free / {len(pro)} pro / {len(backlog)} backlog"}

plan = plan_mvp(["a","b","c","d","e","f"], free_count=2, pro_count=2)
assert plan["free_tier"]  == ["a","b"]
assert plan["pro_tier"]   == ["c","d"]
assert plan["backlog"]    == ["e","f"]
assert "2 free" in plan["summary"]
print("\u2705 plan_mvp correct")

# ─── ContentStore ─────────────────────────────────────────────────────────────
class ContentStore:
    def __init__(self): self._store = {}
    def add(self, uid, p, c):
        cid = secrets.token_urlsafe(8)
        self._store[cid] = {"content_id": cid, "user_id": uid, "prompt": p,
                            "content": c, "created_at": datetime.utcnow().isoformat()+"Z"}
        return cid
    def get(self, cid): return self._store.get(cid)
    def list_user(self, uid): return [v for v in self._store.values() if v["user_id"]==uid]
    def count(self, uid): return sum(1 for v in self._store.values() if v["user_id"]==uid)

store = ContentStore()
cid   = store.add("u1", "prompt", "content")
assert store.get(cid)["content"] == "content"
assert store.count("u1") == 1
assert store.get("bad") is None
print("\u2705 ContentStore correct")

# ─── render_template ──────────────────────────────────────────────────────────
def render_template(template_str, **vars):
    required = set(re.findall(r'\{(\w+)\}', template_str))
    missing  = required - set(vars.keys())
    if missing: raise ValueError(f"Missing template variables: {missing}")
    result = template_str
    for k, v in vars.items(): result = result.replace(f"{{{k}}}", str(v))
    return result

assert render_template("Hello {name}!", name="World") == "Hello World!"
try:
    render_template("Hello {name}!")
    assert False
except ValueError:
    pass
print("\u2705 render_template correct")

# ─── build_core_api ───────────────────────────────────────────────────────────
DAILY_LIMITS = {"free": 5, "pro": 500, "enterprise": float("inf")}
TEMPLATES    = {"email": "Write a {tone} email to {recipient} about {topic}.",
                "tweet": "Write a {tone} tweet about {topic}."}

def check_rate_limit(usage_count, plan):
    limit = DAILY_LIMITS.get(plan, 0)
    if usage_count >= limit: return False, f"Daily limit reached for {plan!r} plan"
    return True, ""

def build_core_api(plan="free", process_fn=None, initial_usage=0):
    app = FastAPI(); store = ContentStore(); state = {"plan": plan, "usage": initial_usage}
    class _G(BaseModel): prompt: str = Field(min_length=1); user_id: str = Field(min_length=1)
    class _T(BaseModel): template: str = Field(min_length=1); vars: dict = {}; user_id: str = Field(min_length=1)
    @app.get("/health")
    def health(): return {"status": "ok", "timestamp": datetime.utcnow().isoformat()+"Z"}
    @app.get("/templates")
    def tmpls(): return {"templates": list(TEMPLATES.keys())}
    @app.post("/generate")
    def gen(req: _G):
        ok, reason = check_rate_limit(state["usage"], state["plan"])
        if not ok: raise HTTPException(429, reason)
        ans = process_fn(req.prompt) if process_fn else req.prompt.upper()
        state["usage"] += 1; cid = store.add(req.user_id, req.prompt, ans)
        return {"content_id": cid, "content": ans, "user_id": req.user_id}
    @app.get("/history/{user_id}")
    def hist(user_id: str):
        items = store.list_user(user_id); return {"user_id": user_id, "count": len(items), "items": items}
    @app.get("/content/{content_id}")
    def get_content(content_id: str):
        item = store.get(content_id)
        if item is None: raise HTTPException(404)
        return item
    return app

c = TestClient(build_core_api(plan="free", process_fn=str.upper), raise_server_exceptions=False)
assert c.get("/health").json()["status"] == "ok"
rg = c.post("/generate", json={"prompt": "hi", "user_id": "u1"})
assert rg.status_code == 200 and rg.json()["content"] == "HI"
assert c.get("/history/u1").json()["count"] == 1
cid2 = rg.json()["content_id"]
assert c.get(f"/content/{cid2}").status_code == 200
assert c.get("/content/bad").status_code == 404
c2 = TestClient(build_core_api(plan="free", process_fn=str.upper, initial_usage=5), raise_server_exceptions=False)
assert c2.post("/generate", json={"prompt": "x", "user_id": "u"}).status_code == 429
print("\u2705 build_core_api correct")

# ─── write_scaffold ───────────────────────────────────────────────────────────
import tempfile
SCAFFOLD_FILES = {
    "app.py": "# scaffold app\n",
    "requirements.txt": "fastapi\nollama\nuvicorn[standard]\nhttpx\n",
    "Procfile": "web: uvicorn app:app --host 0.0.0.0 --port $PORT\n",
}
def write_scaffold(directory):
    base = Path(directory); base.mkdir(parents=True, exist_ok=True)
    for fname, content in SCAFFOLD_FILES.items():
        (base / fname).write_text(content)
    return list(SCAFFOLD_FILES.keys())

with tempfile.TemporaryDirectory() as td:
    files = write_scaffold(td)
    assert len(files) >= 3
    for f in files:
        assert (Path(td) / f).exists()
    assert "fastapi" in (Path(td) / "requirements.txt").read_text()
    assert "uvicorn" in (Path(td) / "Procfile").read_text()
print("\u2705 write_scaffold correct")

print("\nDay 064 \u2014 Capstone Build I complete! \U0001f389")
